## Step 1 — Confirm GPU + log environment

In [ ]:
!nvidia-smi

Mon Dec 15 22:07:23 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
%env VLLM_PLATFORM=cuda


env: VLLM_PLATFORM=cuda


In [ ]:
!pip install -q vllm sglang


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 141.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 127.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 155.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/11

In [ ]:
import vllm
import sglang
print("vLLM OK, SGLang OK")

vLLM OK, SGLang OK


In [ ]:
!fuser -k 8000/tcp || true

In [ ]:
!nohup vllm serve gpt2 \
  --dtype float16 \
  --host 0.0.0.0 \
  --port 8000 \
  > vllm_gpt2.log 2>&1 &


In [ ]:
!tail -n 20 vllm_gpt2.log

W0000 00:00:1765837756.876814    7717 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765837756.876876    7717 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765837756.876879    7717 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765837756.876882    7717 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
2025-12-15 22:29:16.881944: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with 

In [ ]:
!curl --max-time 5 -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8000/health

200


In [ ]:
!curl http://127.0.0.1:8000/health


In [ ]:
import requests
import time

url = "http://127.0.0.1:8000/v1/completions"

payload = {
    "model": "gpt2",
    "prompt": "Explain what a GPU does in one sentence.",
    "max_tokens": 64,
    "temperature": 0.0
}

print("Waiting for vLLM server to start...")
for i in range(24):
    try:
        r = requests.post(url, json=payload, timeout=5)
        r.raise_for_status()
        print("\nStatus:", r.status_code)
        print("Output:")
        print(r.json()["choices"][0]["text"])
        break
    except requests.exceptions.RequestException:
        time.sleep(5)
        if i % 2 == 0:
            print(".", end="")
else:
    print("\nTimed out waiting for server.")


Waiting for vLLM server to start...

Status: 200
Output:


The first thing to do is to define the GPU's state.

The GPU's state is a list of values that are stored in the GPU's registers.

The first value is the current state of the GPU.

The second value is the current state of the GPU.

The


In [ ]:
!tail -n 60 vllm_gpt2.log

2025-12-15 22:29:16.811740: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 22:29:16.830608: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765837756.853295    7717 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765837756.859902    7717 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765837756.876814    7717 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Step 2 - Run this benchmark script (vLLM)

### GPU logging in the background

In [ ]:
# Use higher frequency (100ms) to catch fast GPU spikes for small models like GPT-2
!nohup bash -lc 'nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,memory.total --format=csv -l 0.1 > gpu_util.csv' >/dev/null 2>&1 &
!echo "GPU logging started -> gpu_util.csv (100ms sampling frequency)"

GPU logging started -> gpu_util.csv


### Benchmark script

In [ ]:
import time, requests, pandas as pd

URL = "http://127.0.0.1:8000/v1/completions"
MODEL = "gpt2"

PROMPTS = [
    "Explain cloud computing in one sentence.",
    "Summarize: model serving platforms need to balance latency, throughput, and cost.",
    "Write 3 bullet points comparing CPU vs GPU for inference.",
    "Explain what batching means in inference and why it improves throughput.",
]

def run_once(prompt, max_tokens=128, temperature=0.0):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    t0 = time.time()
    r = requests.post(URL, json=payload, timeout=180)
    t1 = time.time()
    r.raise_for_status()
    data = r.json()
    text = data["choices"][0]["text"]
    usage = data.get("usage", {})
    return {
        "latency_s": t1 - t0,
        "output_text": text,
        "prompt_tokens": usage.get("prompt_tokens", None),
        "completion_tokens": usage.get("completion_tokens", None),
        "total_tokens": usage.get("total_tokens", None),
    }

rows = []

# Warm-up
_ = run_once("Say hello.", max_tokens=32)

for max_toks in [64, 128, 256]:
    for prompt in PROMPTS:
        for trial in range(5):
            out = run_once(prompt, max_tokens=max_toks)
            completion_tokens = out["completion_tokens"]
            approx_tokens = len(out["output_text"].split())
            gen_tokens = completion_tokens if completion_tokens is not None else approx_tokens
            rows.append({
                "framework": "vLLM",
                "model": MODEL,
                "max_tokens": max_toks,
                "prompt_len_chars": len(prompt),
                "trial": trial,
                "latency_s": out["latency_s"],
                "gen_tokens": gen_tokens,
                "tokens_per_sec": gen_tokens / out["latency_s"] if out["latency_s"] > 0 else None,
            })

df_vllm_gpt2 = pd.DataFrame(rows)
df_vllm_gpt2.head()

,framework,model,max_tokens,prompt_len_chars,trial,latency_s,gen_tokens,tokens_per_sec
0,vLLM,gpt2,64,40,0,0.114280,64,560.026904
1,vLLM,gpt2,64,40,1,0.106656,64,600.058245
2,vLLM,gpt2,64,40,2,0.113520,64,563.775402
3,vLLM,gpt2,64,40,3,0.112832,64,567.213427
4,vLLM,gpt2,64,40,4,0.112211,64,570.355333


In [ ]:
df_vllm_gpt2.to_csv("vllm_gpt2_results.csv", index=False)
print("Saved:", "vllm_gpt2_results.csv")

Saved: vllm_gpt2_results.csv


In [ ]:
# Use higher frequency (100ms) for better GPU utilization capture
!nohup bash -lc 'nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,memory.total --format=csv -l 0.1 > gpu_util_gpt2_vllm.csv' >/dev/null 2>&1 &

### Stop GPU logging

In [ ]:
!pkill -f "nvidia-smi --query-gpu" || true
!echo "GPU logging stopped"

^C
GPU logging stopped
